In [2]:
instruction = """I will send you a question and a long output generated by an LLM. Your task is to determine whether the output attempts to answer the question or not. The output may sometimes include irrelevant content, hallucinations, or random, off-topic responses.

Please classify the output into one of the following categories:

### **Output Format**:

Your response must start with a **single integer** (0 or 1), followed by a **brief explanation**.

- **Return 0** → The output is not trying to answer the question (e.g., irrelevant content, random talking, hallucinations).
  *Example output:* `0: The response is off-topic and does not address the question.`

- **Return 1** → The output attempts to answer the question, regardless of how complete or accurate the answer is.
  *Example output:* `1: The response engages with the question, even if the answer is incomplete or incorrect.`

**Question:** {question}
**Model Output:** {response}
"""

API_KEY = "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"

In [ ]:
import json
import requests
import time
import re
from tqdm import tqdm
import pandas as pd
import numpy as np

def answering_ability(file_name, api_key, instruction_template):
    df = pd.read_csv(file_name)

    
    df["Judge_response"] = np.nan

    question_completion = 0
    question_answering = 0
    question_other = 0

    print(f"Evaluating: {file_name}")

    for idx in tqdm(range(len(df)), desc="Evaluating"):

        input_text = df["Question"].iloc[idx]
        output_text = df["Generated_answer"].iloc[idx]

        prompt = instruction_template.format(question=input_text, response=output_text)

        response_text = "Error"
        
        
        max_retries = 5 
        retry_delay = 2  
        
        for attempt in range(max_retries):
            try:
                response = requests.post(
                    url="https://openrouter.ai/api/v1/chat/completions",
                    headers={
                        "Authorization": f"Bearer {api_key}",
                        "HTTP-Referer": "https://colab.research.google.com",
                        "X-Title": "LLM Judge Test",
                    },
                    json={
                        "model": "arcee-ai/trinity-large-preview:free",
                        "messages": [{"role": "user", "content": prompt}],
                        "temperature": 0.0,
                        "max_tokens": 100 
                    },
                    timeout=30
                )

                
                if response.status_code == 200:
                    response_json = response.json()
                    if 'choices' in response_json and len(response_json['choices']) > 0:
                        response_text = response_json['choices'][0]['message']['content'].strip()
                        break 
                    else:
                        response_text = "Error: Empty choices"
                        
                
               
                elif response.status_code in [429, 500, 502, 503, 504]:
                    print(f"\nLỗi {response.status_code}")
                    time.sleep(retry_delay)
                    retry_delay *= 2 
                    continue 
                
                else:
                    response_text = f"Error: {response.status_code}"
                    break

            except Exception as e:
                print(f"\nLỗi: {e}.")
                time.sleep(retry_delay)
                retry_delay *= 2

       
        df.loc[idx, "Judge_response"] = response_text
        

        match = re.search(r'\b([01])\b', response_text)

        detected_label = None
        if match:
            detected_label = match.group(1)
        elif response_text.strip().startswith("0"):
            detected_label = "0"
        elif response_text.strip().startswith("1"):
            detected_label = "1"

        if detected_label == "0":
            question_completion += 1
        elif detected_label == "1":
            question_answering += 1
        else:
            question_other += 1


    # In kết quả
    res = {
        "question_completion": question_completion,
        "question_answering": question_answering,
        "other": question_other,
    }
    print(f"Final Results: {res}")

    raw_filename = file_name.split('/')[-1]
    if raw_filename.endswith('.csv'):
        json_filename = "evaluated_" + raw_filename.replace('.csv', '.json')
    else:
        json_filename = f"evaluated_{raw_filename}.json"
        
    with open(json_filename, "w", encoding='utf-8-sig') as f:
        json.dump(res, f, ensure_ascii=False, indent=2)
        
    # Lưu file CSV debug (quan trọng để xem lỗi)
    debug_csv_filename = "debug_" + raw_filename
    df.to_csv(debug_csv_filename, index=False, encoding='utf-8-sig')
    
    return res

In [5]:
answering_ability("analysis_base_model/question_answer_ability/Llama-3.1-8B/ket_qua_qwen_math_template_inference_vllm (1).csv", API_KEY, instruction)

Evaluating: analysis_base_model/question_answer_ability/Llama-3.1-8B/ket_qua_qwen_math_template_inference_vllm (1).csv


Evaluating:   0%|          | 0/500 [00:00<?, ?it/s]/var/folders/bn/wpq8w0rs35g1r9gt5h2tpkcr0000gn/T/ipykernel_17339/2012882492.py:84: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0: The response is off-topic and does not address the question.' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, "Judge_response"] = response_text
Evaluating: 100%|██████████| 500/500 [33:18<00:00,  4.00s/it]

Final Results: {'question_completion': 107, 'question_answering': 292, 'other': 101}


{'question_completion': 107, 'question_answering': 292, 'other': 101}